In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
import pandas as pd

In [3]:
class MLP(nn.Module):
    def __init__(self, input_size):
        super(MLP, self).__init__()
        #структура нейросети: input_size - 128 - 64 - 32 - 1
        self.layers = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.layers(x)

In [4]:
traffic = pd.read_csv('../../data/df_train_traffic_encoded_scaled.csv')

In [5]:
#делим на 60000, потому что остальные переменные в районе -1 и 1, веса становятся слишком большими и результаты становятся нестабильными
traffic['Трафик'] = traffic['Трафик'] / 60000
traffic

,Трафик,Численность населения,Количество домохозяйств,"Трафик пеший, в час","Трафик авто, в час",month_sin,month_cos,pca_1,pca_2,pca_3,pca_4,Населенный пункт_fold_traffic,Регион_fold_traffic,"Дата открытия, категориальный_Новый","Дата открытия, категориальный_Открыт давно","Дата открытия, категориальный_Средний по возрасту","Торговая площадь, категориальный_Большой","Торговая площадь, категориальный_Маленький","Торговая площадь, категориальный_Очень большой","Торговая площадь, категориальный_Средний"
0,0.994367,-0.183291,-0.634931,-0.658506,0.29525,-8.660254e-01,5.000000e-01,-0.647364,0.219085,0.058661,-1.248180,-0.273791,-0.099160,0,0,1,0,0,0,1
1,0.944567,-0.183291,-0.634931,-0.658506,0.29525,5.000000e-01,-8.660254e-01,-0.647364,0.219085,0.058661,-1.248180,-0.273791,-0.099160,0,0,1,0,0,0,1
2,0.858133,-0.183291,-0.634931,-0.658506,0.29525,5.000000e-01,8.660254e-01,-0.647364,0.219085,0.058661,-1.248180,-0.273791,-0.099160,0,0,1,0,0,0,1
3,0.944883,-0.183291,-0.634931,-0.658506,0.29525,1.224647e-16,-1.000000e+00,-0.647364,0.219085,0.058661,-1.248180,-0.273791,-0.099160,0,0,1,0,0,0,1
4,0.968800,-0.183291,-0.634931,-0.658506,0.29525,-5.000000e-01,-8.660254e-01,-0.647364,0.219085,0.058661,-1.248180,-0.273791,-0.099160,0,0,1,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
232664,0.861267,-0.187406,-0.712807,-0.009117,0.52820,-8.660254e-01,5.000000e-01,-1.678946,0.112034,-0.095371,-0.163764,-1.463815,-0.786469,1,0,0,0,0,0,1
232665,0.858600,-0.187406,-0.712807,-0.009117,0.52820,-5.000000e-01,8.660254e-01,-1.678946,0.112034,-0.095371,-0.163764,-1.463815,-0.786469,1,0,0,0,0,0,1
232666,0.826550,-0.187406,-0.712807,-0.009117,0.52820,-1.000000e+00,-1.836970e-16,-1.678946,0.112034,-0.095371,-0.163764,-1.463815,-0.786469,1,0,0,0,0,0,1
232667,0.868583,-0.187406,-0.712807,-0.009117,0.52820,-2.449294e-16,1.000000e+00,-1.678946,0.112034,-0.095371,-0.163764,-1.463815,-0.786469,1,0,0,0,0,0,1


In [6]:
mlp = MLP(input_size=(traffic.shape[1] - 1))
#функци потерь, сочетающая RMSE и MAE
criterion = nn.HuberLoss(delta=1.0)
optimizer = optim.Adam(mlp.parameters(), lr=0.001)

In [7]:
y_traffic = traffic['Трафик']
x_traffic = traffic.drop(['Трафик'], axis=1)

In [8]:
from sklearn.model_selection import train_test_split

x_train_traffic, x_test_traffic, y_train_traffic, y_test_traffic = train_test_split(x_traffic, y_traffic, test_size=0.2, random_state=598)

In [9]:
#pytorch требует данных в формате тензоров
x_tensor = torch.tensor(x_train_traffic.values, dtype=torch.float32)
y_tensor = torch.tensor(y_train_traffic.values, dtype=torch.float32).reshape(-1, 1)

In [10]:
from torch.utils.data import DataLoader, TensorDataset
dataset = TensorDataset(x_tensor, y_tensor)
loader = DataLoader(dataset, batch_size=100, shuffle=True)

In [11]:
from torch.nn.functional import l1_loss

In [12]:
x_test_tensor = torch.tensor(x_test_traffic.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_traffic.values, dtype=torch.float32).reshape(-1, 1)

In [13]:
best_test_rmse = float('inf')

In [14]:
for epoch in range(300):
    mlp.train()
    for batch_x, batch_y in loader:
        optimizer.zero_grad()
        
        predictions = mlp(batch_x)
        loss = criterion(predictions, batch_y)
        
        loss.backward()
        optimizer.step()

    mlp.eval()    
    with torch.no_grad():
        train_preds = mlp(x_tensor)
        rmse_train = torch.sqrt(torch.mean((train_preds - y_tensor) ** 2)).item() * 60000
        mae_train = l1_loss(train_preds * 60000, y_tensor * 60000).item()
        
        test_preds = mlp(x_test_tensor)
        rmse_test = torch.sqrt(torch.mean((test_preds - y_test_tensor) ** 2)).item() * 60000
        mae_test = l1_loss(test_preds * 60000, y_test_tensor * 60000).item()

        if rmse_test < best_test_rmse:
            best_test_rmse = rmse_test
            torch.save(mlp.state_dict(), 'best_mlp_model_traffic.pth')
        
    print(f"Эпоха {epoch}"
          f" Train RMSE: {rmse_train:7.2f}, MAE: {mae_train:7.2f}"
          f" Test RMSE: {rmse_test:7.2f}, MAE: {mae_test:7.2f}")

Эпоха 0 Train RMSE: 9751.43, MAE: 7408.86 Test RMSE: 9823.81, MAE: 7433.69
Эпоха 1 Train RMSE: 9716.91, MAE: 7363.45 Test RMSE: 9788.25, MAE: 7383.33
Эпоха 2 Train RMSE: 9624.32, MAE: 7293.16 Test RMSE: 9694.55, MAE: 7318.75
Эпоха 3 Train RMSE: 9414.64, MAE: 7213.50 Test RMSE: 9503.06, MAE: 7252.63
Эпоха 4 Train RMSE: 9341.92, MAE: 7096.85 Test RMSE: 9451.39, MAE: 7162.30
Эпоха 5 Train RMSE: 9262.59, MAE: 7135.48 Test RMSE: 9408.95, MAE: 7217.97
Эпоха 6 Train RMSE: 8979.77, MAE: 6849.71 Test RMSE: 9150.71, MAE: 6952.05
Эпоха 7 Train RMSE: 8870.49, MAE: 6824.58 Test RMSE: 9061.76, MAE: 6931.61
Эпоха 8 Train RMSE: 8706.50, MAE: 6669.54 Test RMSE: 8963.61, MAE: 6826.50
Эпоха 9 Train RMSE: 8439.71, MAE: 6448.37 Test RMSE: 8694.04, MAE: 6615.76
Эпоха 10 Train RMSE: 8614.97, MAE: 6537.20 Test RMSE: 8868.88, MAE: 6706.57
Эпоха 11 Train RMSE: 8186.71, MAE: 6280.44 Test RMSE: 8482.18, MAE: 6459.08
Эпоха 12 Train RMSE: 8101.61, MAE: 6223.24 Test RMSE: 8386.73, MAE: 6394.82
Эпоха 13 Train RMSE: 8

In [15]:
mlp.load_state_dict(torch.load('best_mlp_model_traffic.pth'))

C:\Users\Acer\AppData\Local\Temp\ipykernel_30080\170227051.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  mlp.load_state_dict(torch.load('best_mlp_model_traffic.pth'))


<All keys matched successfully>